Filter Cultivos de Abandono Prolongado

# Section 1: Standard GeoPandas Approach

In [13]:
import tempfile
import shutil
import os

In [2]:

import geopandas as gpd

# 1. Define input and output paths (using 'r' for Windows raw strings)
input_zip = r"C:\Users\tglla\OneDrive - Universidad de La Laguna\Desktop\Grafcan\Modulo II\GC_MCultivos_2013.zip"

# 2. Read the shapefile directly from the zip archive
print("Reading zipped shapefile...")
gdf = gpd.read_file(input_zip, encoding='utf-8')


Reading zipped shapefile...


In [4]:
# --- NEW: Enforce CRS (WGS 84 / UTM zone 28N) ---
if gdf.crs is None:
    print("No CRS found. Assigning EPSG:32628...")
    gdf = gdf.set_crs(epsg=32628)
elif gdf.crs.to_epsg() != 32628:
    print(f"Reprojecting from {gdf.crs} to EPSG:32628...")
    gdf = gdf.to_crs(epsg=32628)
else:
    print("CRS is already EPSG:32628. Good to go!")

No CRS found. Assigning EPSG:32628...


In [7]:
# 3. Run the query/filter (Change 'TIPO' and 'Platanera' to your actual data)
print("Filtering data...")
filtered_gdf = gdf[(gdf['Cultivo'] == 'Abandono Prolongado') & (gdf['Area_m2'] >= 8000)& (gdf['Area_m2'] <= 8500)]
print(filtered_gdf.shape)
filtered_gdf.head()

Filtering data...
(183, 11)


,Isla,Fecha,Categoria,Agrupacion,Cultivo,Borde,Diseminado,Regadio,Tecnica,Area_m2,geometry
459,Gran Canaria,01/02/2013,Sin Cultivo,Sin Cultivo,Abandono Prolongado,No,Almendros,No,No,8367.992440,"POLYGON ((450940.591 3097633.464, 450943.467 3..."
1287,Gran Canaria,01/03/2013,Sin Cultivo,Sin Cultivo,Abandono Prolongado,No,No,No,No,8411.456057,"POLYGON ((443152.39 3104434.873, 443171.071 31..."
2314,Gran Canaria,01/10/2012,Sin Cultivo,Sin Cultivo,Abandono Prolongado,No,No,No,No,8185.355387,"POLYGON ((457047.49 3079970.162, 457046.338 30..."
3637,Gran Canaria,02/05/2013,Sin Cultivo,Sin Cultivo,Abandono Prolongado,No,No,No,No,8358.872941,"POLYGON ((444629.97 3091742.202, 444616.868 30..."
4162,Gran Canaria,02/10/2012,Sin Cultivo,Sin Cultivo,Abandono Prolongado,No,No,No,No,8412.240626,"POLYGON ((458729.224 3090234.894, 458723.583 3..."


In [ ]:
# print(os.environ['OneDrive'])

In [ ]:
# 4. Save the resulting dataset to a new shapefile
print("Saving to new shapefile...")
output_shp = r"C:\Users\tglla\OneDrive - Universidad de La Laguna\Desktop\Grafcan\Modulo II\parcelas_abandono_prolongado1\parcelas_abandono_prolongado.shp"

filtered_gdf.to_file(output_shp, encoding='utf-8')

print("GeoPandas process complete!")

In [16]:
# Notice we DO NOT add the .zip extension here! shutil.make_archive adds it automatically.
output_zip_base = os.path.join(os.environ['OneDrive'],r"Desktop\Grafcan\Modulo II\capa1")

# Create an invisible temporary folder
with tempfile.TemporaryDirectory() as temp_dir:
    # 1. Define the shapefile name inside the temporary folder
    temp_shp = os.path.join(temp_dir, "capa1.shp")

    # 2. Save the shapefile (and all its auxiliary files) into the temp folder
    filtered_gdf.to_file(temp_shp, encoding='utf-8')

    # 3. Zip the contents of the temp folder into our final destination!
    shutil.make_archive(output_zip_base, 'zip', temp_dir)

print(f"Success! All auxiliary files cleanly zipped into: {output_zip_base}.zip")

Success! All auxiliary files cleanly zipped into: C:\Users\tglla\OneDrive - Universidad de La Laguna\Desktop\Grafcan\Modulo II\capa1.zip


In [ ]:
# CELL 2: QGIS/OSGeo4W Command Line (Direct to Zip)

# 1. We specify -f "ESRI Shapefile" to ensure it knows the format.
# 2. We use /vsizip/ on the output to create the capa1.zip archive.
# 3. We tell it to name the files inside the archive "capa1.shp".

!ogr2ogr -f "ESRI Shapefile" -t_srs EPSG:32628 --config SHAPE_ENCODING UTF-8 -lco ENCODING=UTF-8 -where "TIPO = 'Platanera'" /vsizip/"C:\Users\tglla\OneDrive - Universidad de La Laguna\Desktop\Grafcan\Modulo II\capa1.zip"/capa1.shp /vsizip/"C:\Users\tglla\OneDrive - Universidad de La Laguna\Desktop\Grafcan\Modulo II\GC_MCultivos_2013.zip"

print("QGIS/OGR command line process complete! capa1.zip created.")